# SEER GBM 描述性分析

基于清洗后的 SEER 数据，进行患者基线特征描述和可视化。

分析内容:
1. 人口学特征分布 (性别/年龄/种族)
2. 时间趋势 (诊断年份)
3. 生存时间分布
4. Table 1: 基线特征表 (按生存状态分层)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns

# 全局绘图设置
mpl.rcParams.update({
    'font.family': 'Arial',
    'font.size': 10,
    'axes.linewidth': 0.8,
    'axes.unicode_minus': False,
    'figure.dpi': 300,
    'savefig.dpi': 300,
    'savefig.format': 'tiff',
    'savefig.bbox': 'tight',
    'axes.spines.top': False,
    'axes.spines.right': False,
})
COLORS = ['#5B9BD5', '#ED7D31', '#70AD47', '#FFC000', '#9B59B6', '#44546A']
sns.set_style('white')
plt.rcParams['font.family'] = 'Arial'
plt.rcParams['font.size'] = 11

In [ ]:
# 读取清洗后数据
df = pd.read_csv('cleaned_data.csv')
print(df.columns.tolist())

In [ ]:
# ===== Fig 1: 性别分布 =====
# Sex: 0=male, 1=female (编码自SEER原始值 Male/Female)
fig, ax = plt.subplots()
sex_labels = {0: 'Male', 1: 'Female'}
counts = df['Sex'].map(sex_labels).value_counts()

ax.barh(counts.index, counts.values, color=COLORS[:2], 
        edgecolor='black', linewidth=0.5, height=0.5)

for i, v in enumerate(counts.values):
    ax.text(v + 100, i, f'{v}', va='center', fontsize=10)

ax.set_xlabel('Number of Patients')
ax.set_title('Sex Distribution', fontweight='bold', pad=10)
ax.set_xlim(0, counts.max() * 1.15)
plt.tight_layout()
fig.savefig('fig1_sex.tiff')
plt.close()

In [ ]:
# ===== Fig 2: 年龄分布 =====
# Age Group: <60 / 60-69 / ≥70 (由Age recode合并而来)
fig, ax = plt.subplots()
order = ['<60', '60-69', '≥70']
sns.countplot(x='Age Group', data=df, order=order, ax=ax, 
              palette=COLORS[:3], edgecolor='black', linewidth=0.5)

for p in ax.patches:
    ax.annotate(f'{int(p.get_height())}', 
                (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='bottom', fontsize=10)

ax.set_xlabel('Age Group')
ax.set_ylabel('Number of Patients')
ax.set_title('Age Distribution', fontweight='bold', pad=10)
plt.tight_layout()
fig.savefig('fig2_age.tiff')
plt.close()

In [ ]:
# ===== Fig 3: 种族分布 =====
# Race: White / Black / Asian or Pacific Islander / American Indian/Alaska Native
# 来自 SEER Race recode (W, B, AI, API)，已排除 Unknown
fig, ax = plt.subplots()
counts = df['Race'].value_counts()

ax.barh(counts.index, counts.values, color=COLORS[:len(counts)], 
        edgecolor='black', linewidth=0.5, height=0.5)

for i, v in enumerate(counts.values):
    ax.text(v + 100, i, f'{v}', va='center', fontsize=10)

ax.set_xlabel('Number of Patients')
ax.set_title('Race Distribution', fontweight='bold', pad=10)
ax.set_xlim(0, counts.max() * 1.15)
plt.tight_layout()
fig.savefig('fig3_race.tiff')
plt.close()

In [ ]:
# ===== Fig 4: 年度发病趋势 =====
# Year of diagnosis: SEER原始变量，2004-2019
fig, ax = plt.subplots()
year_counts = df['Year of diagnosis'].value_counts().sort_index()

ax.plot(year_counts.index, year_counts.values, 
        color=COLORS[0], linewidth=2, marker='o', markersize=4)

ax.set_xlabel('Year of Diagnosis')
ax.set_ylabel('Number of Patients')
ax.set_title('Annual GBM Incidence Trend (2004-2019)', fontweight='bold', pad=10)
plt.tight_layout()
fig.savefig('fig4_year_trend.tiff')
plt.close()

In [ ]:
# ===== Fig 5: 生存时间分布 =====
# Survival months: SEER SRV_TIME_MON，已排除 NA 和 ≤0
fig, ax = plt.subplots()
ax.hist(df['Survival months'], bins=50, color=COLORS[0], 
        edgecolor='black', linewidth=0.5)

ax.set_xlabel('Survival Months')
ax.set_ylabel('Number of Patients')
ax.set_title('Distribution of Survival Time', fontweight='bold', pad=10)
plt.tight_layout()
fig.savefig('fig5_survival.tiff')
plt.close()

In [ ]:
# ===== Table 1: 基线特征表 =====
# 按 Vital status recode 分层: Alive(0) vs Dead(1)
import numpy as np
alive = df[df['Vital status recode'] == 0]
dead = df[df['Vital status recode'] == 1]
variables = ['Age Group', 'Sex', 'Race', 'Grade', 'Stage', 
             'Surgery', 'Radiation', 'Chemotherapy']

rows = []
for var in variables:
    alive_counts = alive[var].value_counts()
    alive_total = len(alive)
    
    dead_counts = dead[var].value_counts()
    dead_total = len(dead)
    
    all_counts = df[var].value_counts()
    all_total = len(df)
    
    for cat in all_counts.index:
        n_all = all_counts[cat]
        pct_all = n_all / all_total * 100
        n_alive = alive_counts.get(cat, 0)
        pct_alive = n_alive / alive_total * 100
        n_dead = dead_counts.get(cat, 0)
        pct_dead = n_dead / dead_total * 100
        
        rows.append({
            'Variable': var if cat == all_counts.index[0] else '',
            'Category': cat,
            'Total (N={}, %)'.format(all_total): f'{n_all} ({pct_all:.1f})',
            'Alive (N={}, %)'.format(alive_total): f'{n_alive} ({pct_alive:.1f})',
            'Dead (N={}, %)'.format(dead_total): f'{n_dead} ({pct_dead:.1f})'
        })

table1 = pd.DataFrame(rows)
table1.to_csv('table1.csv', index=False)
print(table1.to_string(index=False))